# Experiment C — QAOA на GPU-ускоренном Qiskit Aer

**Запускается на суперкомпьютере НИУ ВШЭ**, так как требуется NVIDIA GPU с cuQuantum для statevector-симуляции QAOA на N=16-24 кубитах.

## Подготовка на кластере

```bash
# На login-узле:
conda env create -f environment.yml
conda activate cif
pip install qiskit-aer-gpu cuquantum-python
```

Требуемые файлы (перенести через scp/sftp с локальной машины):
- `pyproject.toml`, `environment.yml`
- `src/cif/` (весь пакет)
- `data/processed/sp500_prices.parquet`, `data/processed/moex_prices.parquet`

## Что делает notebook

1. Загружает реальные SP500/MOEX данные.
2. Проверяет наличие GPU backend для Qiskit Aer.
3. Запускает QAOA с p ∈ {1, 2, 3} на N = {6, 10, 16, 20, 24} активов — CPU и GPU режимы.
4. Сравнивает approximation ratio к brute force (где N ≤ 12) и к `neal` SA (при N > 12).
5. Строит scalability plot QAOA CPU vs QAOA GPU.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cif.problem import PortfolioProblem
from cif.data.statistics import log_returns, annualised_mu, annualised_sigma
from cif.classical.brute_force import brute_force_cardinality_continuous
from cif.classical.milp import solve_miqp_scip
from cif.quantum.neal_sampler import solve_with_neal_selection
from cif.quantum.qiskit_qaoa import run_qaoa_selection, QaoaConfig

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 110

## GPU availability check

In [ ]:
from qiskit_aer import AerSimulator

try:
    gpu_sim = AerSimulator(method='statevector', device='GPU')
    print('GPU backend available:', gpu_sim.configuration().description)
    HAS_GPU = True
except Exception as e:
    print(f'GPU backend not available: {e}')
    print('Falling back to CPU-only run')
    HAS_GPU = False

cpu_sim = AerSimulator(method='statevector', device='CPU')
print('CPU backend:', cpu_sim.configuration().description)

## Загрузка данных

In [ ]:
prices = pd.read_parquet('../data/processed/sp500_prices.parquet')
train = prices.loc['2015':'2024']
rets = log_returns(train)
mu_full = annualised_mu(rets).values
sigma_full = annualised_sigma(rets, method='ledoit_wolf').values
names_full = tuple(rets.columns)

print(f'{len(names_full)} tickers, {len(rets)} days')
print(f'mu range: [{mu_full.min():.3f}, {mu_full.max():.3f}]')

## QAOA scalability on CPU and GPU

Для каждого N ∈ {6, 10, 16, 20, 24} запустим QAOA на CPU (до 14 кубит) и на GPU (до 24 кубит). p ∈ {1, 2, 3}. Сравним approximation ratio и wall time.

In [ ]:
N_values = [6, 10, 16, 20, 24]
p_values = [1, 2, 3]
K_ratio = 0.4  # K = 0.4 * N

results_qaoa = []

for N in N_values:
    K = max(2, int(round(K_ratio * N)))
    mu = mu_full[:N]
    sigma = sigma_full[:N, :N]
    names = names_full[:N]
    problem = PortfolioProblem(
        mu=mu, sigma=sigma, asset_names=names,
        risk_aversion=2.0, cardinality=K,
    )

    # Ground truth
    if N <= 12:
        gt = brute_force_cardinality_continuous(problem, cardinality=K)
        gt_label = 'brute_force'
    else:
        gt = solve_miqp_scip(problem, time_limit_s=60)
        gt_label = 'scip_miqp'
    print(f'N={N} K={K}: ground truth ({gt_label}) obj={gt.objective:.5f}')

    # neal as quantum-inspired reference
    obj_scale = float(np.abs(mu).max())/K + 2.0 * float(np.abs(sigma).max())/(K*K)
    cp = 100.0 * max(obj_scale, 1e-6)
    neal_sol = solve_with_neal_selection(
        problem, num_reads=1000, num_sweeps=500, seed=42,
        top_subsets=100, cardinality_penalty=cp,
    )
    print(f'  neal: obj={neal_sol.objective:.5f}, gap={100*(neal_sol.objective - gt.objective)/abs(gt.objective):+.2f}%, time={neal_sol.wall_time_s:.2f}s')

    devices = ['CPU']
    if HAS_GPU and N >= 10:
        devices.append('GPU')
    if N > 14 and 'CPU' in devices:
        # CPU statevector becomes too slow
        devices = ['GPU'] if HAS_GPU else []

    for p in p_values:
        for device in devices:
            try:
                cfg = QaoaConfig(p=p, shots=4096, optimizer_maxiter=60, device=device, seed=42)
                sol = run_qaoa_selection(problem, config=cfg)
                ratio = sol.objective / gt.objective if gt.objective < 0 else float('nan')
                gap = 100 * (sol.objective - gt.objective) / abs(gt.objective)
                results_qaoa.append({
                    'N': N, 'K': K, 'p': p, 'device': device,
                    'wall_time_s': sol.wall_time_s,
                    'objective': sol.objective,
                    'approx_ratio': ratio,
                    'gap_pct': gap,
                    'gt_obj': gt.objective,
                    'gt_label': gt_label,
                })
                print(f'  QAOA {device} p={p}: obj={sol.objective:.5f}, gap={gap:+.2f}%, time={sol.wall_time_s:.2f}s')
            except Exception as e:
                print(f'  QAOA {device} p={p}: FAILED {type(e).__name__}: {str(e)[:100]}')
                results_qaoa.append({
                    'N': N, 'K': K, 'p': p, 'device': device,
                    'wall_time_s': float('nan'), 'objective': float('nan'),
                    'approx_ratio': float('nan'), 'gap_pct': float('nan'),
                    'error': str(e)[:200],
                })

df_qaoa = pd.DataFrame(results_qaoa)
df_qaoa

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for device, marker in [('CPU', 'o'), ('GPU', '^')]:
    sub = df_qaoa[df_qaoa['device'] == device].dropna(subset=['wall_time_s'])
    for p_val in sorted(sub['p'].unique()):
        sub_p = sub[sub['p'] == p_val].sort_values('N')
        if not sub_p.empty:
            axes[0].plot(sub_p['N'], sub_p['wall_time_s'], marker=marker, label=f'{device} p={p_val}')
axes[0].set_yscale('log')
axes[0].set_xlabel('N (active qubits)')
axes[0].set_ylabel('Wall time (s, log)')
axes[0].set_title('QAOA wall time vs N')
axes[0].legend()
axes[0].grid(alpha=0.3)

for device, marker in [('CPU', 'o'), ('GPU', '^')]:
    sub = df_qaoa[df_qaoa['device'] == device].dropna(subset=['gap_pct'])
    for p_val in sorted(sub['p'].unique()):
        sub_p = sub[sub['p'] == p_val].sort_values('N')
        if not sub_p.empty:
            axes[1].plot(sub_p['N'], sub_p['gap_pct'], marker=marker, label=f'{device} p={p_val}')
axes[1].set_xlabel('N (active qubits)')
axes[1].set_ylabel('Gap to ground truth (%)')
axes[1].set_title('QAOA solution quality vs N')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/final/figures/exp_c_qaoa_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

## Save results and print business-facing summary

In [ ]:
df_qaoa.to_csv('../results/experiment_c_qaoa.csv', index=False)
print('Saved to results/experiment_c_qaoa.csv')

print('\n=== Experiment C summary ===')
print(f'Total QAOA runs: {len(df_qaoa)}')
print(f'Max N on CPU: {df_qaoa[df_qaoa["device"]=="CPU"]["N"].max() if "CPU" in df_qaoa["device"].values else "n/a"}')
print(f'Max N on GPU: {df_qaoa[df_qaoa["device"]=="GPU"]["N"].max() if "GPU" in df_qaoa["device"].values else "n/a"}')
success = df_qaoa.dropna(subset=["gap_pct"])
print(f'\nBest approximation ratio per N:')
if not success.empty:
    best = success.loc[success.groupby('N')['gap_pct'].idxmin()]
    print(best[['N', 'K', 'p', 'device', 'gap_pct', 'wall_time_s']].to_string(index=False))